# Regression Example with EOTorch

This notebook demonstrates how to train a pixel-wise regression model using EOTorch's `RegressionTask` for geospatial prediction tasks.

Common regression applications include:
- **Elevation/DEM prediction**: Predicting digital elevation models from satellite imagery
- **Biomass estimation**: Predicting forest biomass from multispectral data
- **Precipitation prediction**: Inferring rainfall from satellite observations
- **Temperature mapping**: Predicting surface temperature from thermal bands
- **Chlorophyll concentration**: Estimating vegetation indices from spectral data

The notebook is structured as follows:
1. Importing functions and libraries
2. Defining variables
3. Normalization and generating labels
4. Preprocessing -- Route A: patches generated and saved to disk
5. Preprocessing -- Route B: patches sampled on the fly
6. Model setup and training -- Route A: patch-based
7. Model setup and training -- Route B: on-the-fly
8. Inference

Routes A and B are alternatives to each other -- pick one. Section 4 pairs with section 6, and section 5 pairs with section 7.

## 1) Importing functions and libraries

In [ ]:
from pathlib import Path

import wandb
from lightning.pytorch.loggers import WandbLogger
from lightning import seed_everything
from lightning.pytorch import Trainer
from lightning.pytorch.callbacks import (
    EarlyStopping,
    LearningRateMonitor,
    ModelCheckpoint,
)

from eotorch.data import (
    PatchDataModule,
    RegressionDataModule,
    RegressionTask,
    get_regression_dataset,
    splits,
)
from eotorch.processing import generate_patches_from_files, normalize


## 2) Defining variables

Paths to your raw imagery and label rasters, plus the hyperparameters referenced throughout the rest of the notebook.

In [ ]:
# You'll have to adjust the paths according to your setup
DATASET_DIR = Path.home()
IMG_DIR = DATASET_DIR / "sr_data"
LABEL_DIR = DATASET_DIR / "labels"  # Continuous-valued GeoTIFFs (e.g. elevation, biomass)
NORM_DIR = DATASET_DIR / "normalized"

# Model / data hyperparameters
IN_CHANNELS = 6  # Number of image bands fed into the model
NUM_OUTPUTS = 1  # Number of continuous targets predicted per pixel
PATCH_SIZE = 256
BATCH_SIZE = 16
MAX_EPOCHS = 150


## 3) Normalization and generating labels

### Normalization

Normalize the input satellite imagery to improve model training. This step is optional but highly recommended, and is shared by both preprocessing routes below.

In [ ]:
# Normalize all images in the directory
for f in IMG_DIR.rglob("*.tif"):
    print(f"Processing {f}")
    out_file = NORM_DIR / f.parent.name / f.name
    out_file.parent.mkdir(exist_ok=True, parents=True)
    normalize(
        img_path=f,
        limits=(0.1, 99.5),  # Use 0.1% and 99.5% percentiles for clipping
        out_path=out_file,
        sample_size=0.2,  # Sample 20% of pixels for statistics
    )


### Labels

`RegressionTask` expects one (or more) continuous-valued GeoTIFF(s) per label file, aligned with the imagery in `LABEL_DIR` (e.g. an elevation model, a biomass raster, interpolated field measurements, ...). Unlike segmentation, there is no fixed class palette to define here.

If your label data doesn't already exist as a raster (e.g. it's point observations or vector polygons with a continuous attribute), rasterize it onto a grid matching your imagery before continuing. `eotorch.processing.rasterize` is built for categorical class rasters (`uint8` labels), so for continuous values, rasterize with `rasterio.features.rasterize` (or similar) directly, writing the continuous attribute values instead of class indices.

With normalized images and aligned label rasters in place, pick one of the two preprocessing routes below (they're alternatives, not both needed) -- and use the matching model-setup-and-training section later in the notebook.

***
## 4A) Preprocessing -- generate and save patches locally

`eotorch.processing.patch.generate_patches_from_files` pre-cuts fixed-size feature/label patch pairs from your normalized images and label rasters, and writes them to disk. Training then reads them back through `eotorch.data.PatchDataModule` (section 6).

`DatasetFromPatches` (used internally by `PatchDataModule`) infers whether a label should be cast to `long` or `float` from the label patch's own on-disk dtype -- integer dtypes (segmentation class indices) become `long`, floating dtypes (regression targets, like here) stay `float`. So this route works correctly for continuous regression targets, as long as your label rasters are stored with a floating dtype before patching (they are here, since we skip any class rasterization step).

Splitting into `TRAIN_PATCH_DIR` / `VAL_PATCH_DIR` happens by *image file* below, so patches from the same scene never end up in both the train and validation sets.

Note: `generate_patches_from_files` always writes patches suffixed `_feature.tiff` / `_label.tiff`. If you use a different patch generation script, adjust `image_suffix` / `label_suffix` in section 6's `PatchDataModule` call to match -- otherwise `PatchDataModule` will silently find zero patches.

In [ ]:
TRAIN_PATCH_DIR = DATASET_DIR / "patches" / "train"
VAL_PATCH_DIR = DATASET_DIR / "patches" / "val"

# Hold out one or more full scenes for validation, so patches from the same
# scene never leak between the train and validation sets.
val_img_files = {str(NORM_DIR / "31TFL" / "20230731.tif")}

for f in NORM_DIR.rglob("*.tif"):
    label_path = LABEL_DIR / f.relative_to(NORM_DIR)  # adjust to match your label naming convention
    out_dir = VAL_PATCH_DIR if str(f) in val_img_files else TRAIN_PATCH_DIR

    generate_patches_from_files(
        img_path=f,
        label_path=label_path,
        out_dir=out_dir,
        patch_size=PATCH_SIZE,
        stride=2,  # 50% overlap
        empty_threshold=0.8,  # skip patches that are >20% no-data
    )


## 4B) Preprocessing -- generate patches on the fly

This route mirrors `segmentation.ipynb`: `get_regression_dataset` builds a `torchgeo`-based dataset directly over your full-size normalized images and label rasters, and patches are sampled from it dynamically during training by `RegressionDataModule` (section 7) -- no patch files are written to disk, and there's nothing to re-cut if your source rasters change.

### Create the dataset and visualize samples

If your dataset has a temporal component, you can match image and label files via filenames, the same way as for segmentation:

```python
image_filename_regex=r'.*_(?P<date>\d{4})_.*',
label_filename_regex=r'.*_(?P<date>\d{4})_.*',
date_format="%Y",
```

Use `plot_samples()` to inspect random patches and validate that images are correctly matched with labels.

In [ ]:
ds = get_regression_dataset(
    images_dir=NORM_DIR,
    labels_dir=LABEL_DIR,
    image_filename_regex=r"(?P<date>\d{8})\.tif",
    label_filename_regex=r".*(?P<start>\d{4})_(?P<stop>\d{4}).tif",
    image_date_format="%Y%m%d",
    label_date_format="%Y",
    all_image_bands=("B02", "B03", "B04", "B08", "B11", "B12"),
    rgb_bands=("B04", "B03", "B02"),
    cache_size=0,
)
ds.plot_samples(n=2, patch_size=PATCH_SIZE, show_filepaths=True, dataset_type="regression")


#### Splitting your dataset

We always need a validation set for training. Either split your data yourself, or just provide your entire dataset as the `train_dataset` -- `RegressionDataModule` will then perform a default file-wise train/val split for you.

#### File-based split:

Simply specify one or multiple files of your dataset that should be moved to the validation (or test) set.

In [ ]:
train_ds, val_ds = splits.file_wise_split(
    dataset=ds,
    val_img_files="/path/to/one/of/your/normalized/images.tif",
)


#### AOI-based split (alternative):

Define an AOI for the splits, either via a geojson file (recommended) or explicit bounds. You can also specify a `buffer_size_metres` to avoid overlap (due to patch size) between your datasets.

```python
train_ds, val_ds = splits.aoi_split(
    dataset=ds,
    aoi_files="/path/to/val_data.geojson",
    buffer_size_metres=1000,
)
```

***
## 5A) Model setup and training -- patch-based

Uses the patches generated in section 4 via `PatchDataModule`, which yields plain `(image, label)` tensor pairs rather than the `torchgeo` sample dicts used by Route B. Structured close to a typical patch-based training script, with the model/loss swapped for `RegressionTask`.

Only run this section if you generated patches in section 4. If you're using Route B, skip to section 7.

In [ ]:
RUN_NAME = "regression_patchbased_v1"
CHECKPOINT_DIR = Path("../ml/checkpoints")

dm = PatchDataModule(
    TRAIN_PATCH_DIR,
    VAL_PATCH_DIR,
    batch_size=BATCH_SIZE,
    # transform=Augment(),
    image_suffix="feature",
    label_suffix="label",
)

model = RegressionTask(
    model="unet",  # Architecture options: 'unet', 'unet++'
    backbone="resnet50",  # Encoder: 'resnet18', 'resnet34', 'resnet50', etc.
    in_channels=IN_CHANNELS,
    num_outputs=NUM_OUTPUTS,
    loss="mse",  # Loss function: 'mse', 'mae', 'huber', 'smoothl1'
    lr=1e-3,
    freeze_backbone=False,  # Set True to fine-tune with a frozen encoder - requires 3-channel input (RGB)
)

# Optional experiment tracking -- drop `logger=wandb_logger` below (and this
# block) if you don't use Weights & Biases.
wandb_logger = WandbLogger(name=RUN_NAME, project="PEI")
wandb_logger.experiment.config["batch_size"] = BATCH_SIZE

trainer = Trainer(
    max_epochs=MAX_EPOCHS,
    accelerator="gpu",
    devices=1,
    logger=wandb_logger,
    callbacks=[
        LearningRateMonitor(logging_interval="epoch"),
        ModelCheckpoint(dirpath=CHECKPOINT_DIR, filename=RUN_NAME, save_last=True),
    ],
)

ckpt_path = CHECKPOINT_DIR / f"{RUN_NAME}.ckpt"
trainer.fit(
    model,
    datamodule=dm,
    ckpt_path=str(ckpt_path) if ckpt_path.is_file() else None,
)
wandb.finish()

## 5B) Model setup and training -- on-the-fly

Uses the dataset(s) built in section 5 via `RegressionDataModule`, which samples patches dynamically at train time and yields `torchgeo` sample dicts (`{"image": ..., "mask": ...}`).

Only run this section if you're using Route B. If you generated patches in section 4 instead, use section 6.

### Data module

Use `train_sampler_config` to change how patches are sampled from your train dataset (default: `GridGeoSampler` with a stride of `patch_size / 2`). See https://torchgeo.readthedocs.io/en/stable/api/samplers.html for available samplers.

- **Out of memory (RAM)**: Decrease `num_workers` and/or `cache_size` (set on the dataset).
- **Out of GPU memory**: Reduce `batch_size`.
- **Low GPU utilization**: Increase `num_workers`.

In [ ]:
seed_everything(42, workers=False)

module = RegressionDataModule(
    train_dataset=train_ds,
    val_dataset=val_ds,
    batch_size=BATCH_SIZE,
    patch_size=PATCH_SIZE,
    pin_memory=True,
    # num_workers=6,  # This doesn't work on Windows
)

Preview the train/val samplers before committing to a long training run -- there should be no overlap between the splits.

In [ ]:
module.preview_data_sampling(max_samples=100)

### Task, trainer, and training run

`num_outputs` sets how many continuous values are predicted per pixel (1 for a single target like elevation, more for multi-target regression). See `RegressionTask` for all supported `model`/`backbone`/`loss` options.

In [ ]:
task = RegressionTask(
    model="unet",  # Architecture options: 'unet', 'unet++'
    backbone="resnet50",  # Encoder: 'resnet18', 'resnet34', 'resnet50', etc.
    in_channels=IN_CHANNELS,
    num_outputs=NUM_OUTPUTS,
    loss="mse",  # Loss function: 'mse', 'mae', 'huber', 'smoothl1'
    lr=1e-3,
    freeze_backbone=False,  # Set True to fine-tune with a frozen encoder - requires 3-channel input (RGB)
)

# Optional experiment tracking -- drop `logger=wandb_logger` below (and this
# block) if you don't use Weights & Biases.
wandb_logger = WandbLogger(name=RUN_NAME, project="PEI")
wandb_logger.experiment.config["batch_size"] = BATCH_SIZE

trainer = Trainer(
    callbacks=[
        LearningRateMonitor(logging_interval="epoch"),
        EarlyStopping(
            monitor="val_loss",
            patience=20,
            mode="min",
        ),
        ModelCheckpoint(
            save_last=True,
            monitor="val_loss",
            save_top_k=5,
            filename="{epoch}-{val_loss:.4f}-{val_MAE:.4f}",
            mode="min",
        ),
    ],
    max_epochs=MAX_EPOCHS,
    logger=wandb_logger,
    devices=1,
    accelerator="gpu"
)

trainer.fit(model=task, datamodule=module)
wandb.finish()

***
## 6) Inference

Use the trained model to generate continuous predictions (e.g. elevation maps, biomass estimates) across full GeoTIFF files. `RegressionTask.predict_on_tif_file`:

1. Loads the checkpoint and reconstructs the model + hyperparameters.
2. Reads `patch_size` from the checkpoint's saved datamodule hyperparameters, if present.
3. Tiles the input image into overlapping patches, predicts each, and stitches them back together.
4. Saves the result to a GeoTIFF with the original CRS and georeferencing.

In [ ]:
# Get the best checkpoint from training
best_model_path = trainer.checkpoint_callback.best_model_path
print(f"Using weights from: {best_model_path}")

# Define output path for predictions
out_file_path = "/path/to/predictions/elevation_predictions.tif"

# Run inference on a full GeoTIFF
RegressionTask.predict_on_tif_file(
    tif_file_path="/path/to/input/image.tif",
    checkpoint_path=best_model_path,
    out_file_path=out_file_path,
    show_results=False,
    batch_size=32,  # Adjust based on GPU memory
)
